# 📊 Notebook 1 — EDA & Text Preprocessing

**Breast Ultrasound Report Generation | MSc AI — University of East London**

This notebook covers:
1. Loading and inspecting the BrEaST-Lesions CSV dataset
2. Building natural-language input/target text pairs from clinical columns
3. Exploratory Data Analysis (word cloud, length distribution, n-grams, LDA topics)
4. Named Entity Recognition and readability analysis

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import spacy
import textstat
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 1. Load & Inspect Dataset

In [ ]:
from src.preprocessing.data_loader import load_and_build_reports

# Builds input_text + target_text from CSV clinical columns
df = load_and_build_reports()
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
print(f"Total records      : {len(df)}")
print(f"Unique input texts : {df['input_text'].nunique()}")
print(f"Avg report length  : {df['input_text'].str.len().mean():.1f} characters")
print(f"\nSample input text:\n{df['input_text'].iloc[0]}")
print(f"\nSample target text:\n{df['target_text'].iloc[0]}")

## 2. Word Cloud

In [ ]:
text = ' '.join(df['input_text'].dropna())
wc = WordCloud(width=800, height=600, background_color='white',
               colormap='Blues', max_words=100).generate(text)

plt.figure(figsize=(10, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud — Breast Ultrasound Reports', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/wordcloud.png', dpi=150)
plt.show()

## 3. Report Length Distribution

In [ ]:
df['report_length'] = df['input_text'].str.len()
mean_val = df['report_length'].mean()

plt.figure(figsize=(10, 6))
sns.histplot(df['report_length'], kde=True, color='skyblue')
plt.axvline(mean_val, color='red', linestyle='--')
plt.text(mean_val + 5, plt.ylim()[1] * 0.85, f'Mean: {mean_val:.1f}', color='red', fontsize=12)
plt.title('Distribution of Ultrasound Report Lengths', fontsize=14, fontweight='bold')
plt.xlabel('Length (characters)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(True, linestyle='--', linewidth=0.5)
sns.despine()
plt.tight_layout()
plt.savefig('../results/figures/report_length_distribution.png', dpi=150)
plt.show()

## 4. Top-10 Unigrams, Bigrams, Trigrams

In [ ]:
def get_top_ngrams(corpus, n=2, top=10):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(corpus)
    bow = vec.transform(corpus)
    freq = sorted(
        [(w, bow.sum(axis=0)[0, i]) for w, i in vec.vocabulary_.items()],
        key=lambda x: x[1], reverse=True
    )[:top]
    return freq

for n, label in [(1,'Unigram'), (2,'Bigram'), (3,'Trigram')]:
    top = get_top_ngrams(df['input_text'].dropna(), n=n)
    top_df = pd.DataFrame(top, columns=[label, 'Frequency'])

    plt.figure(figsize=(10, 5))
    if n == 1:
        sns.barplot(x=label, y='Frequency', data=top_df, palette='Blues_d')
    else:
        sns.barplot(x='Frequency', y=label, data=top_df, palette='Blues_d')
    plt.title(f'Top 10 {label}s in Reports', fontsize=14, fontweight='bold')
    plt.grid(True, linestyle='--', linewidth=0.5)
    sns.despine()
    plt.tight_layout()
    plt.savefig(f'../results/figures/top_{n}grams.png', dpi=150)
    plt.show()

## 5. LDA Topic Modelling

In [ ]:
corpus = df['input_text'].dropna()
vec    = CountVectorizer(stop_words='english').fit(corpus)
bow    = vec.transform(corpus)
lda    = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(bow)

feat_names = vec.get_feature_names_out()
print('LDA Topics:')
for i, topic in enumerate(lda.components_):
    words = ' | '.join([feat_names[j] for j in topic.argsort()[:-11:-1]])
    print(f'  Topic {i}: {words}')

## 6. Readability Analysis

In [ ]:
sample = df['input_text'].dropna().iloc[0]
print('Sample text:', sample)
print(f'\nFlesch Reading Ease : {textstat.flesch_reading_ease(sample):.1f}')
print(f'Gunning Fog Index   : {textstat.gunning_fog(sample):.1f}')
print('\n(Lower Flesch = harder to read; Higher Fog = more complex vocabulary)')

## 7. Named Entity Recognition (spaCy)

In [ ]:
nlp  = spacy.load('en_core_web_sm')
doc  = nlp(df['input_text'].dropna().iloc[0])
spacy.displacy.render(doc, style='ent', jupyter=True)